# Ataque Cardiaco con Máquinas de Vectores de Soporte
## Ver 3.0
 En esta versión vamos a incluir el proceso de calibración del hiperparámetro y el uso de kernel

In [76]:
# Importar las librerias

# Cargamos nuestras librerias
import numpy as np
import pandas as pd

# Entorno SciKit Learn
from sklearn.svm import SVC   # Algoritmo incorpora el truco del Kernel.

from sklearn.model_selection import train_test_split  # Dividir el conjunto de prueba y entrenamiento
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_score # Validación cruzada
from sklearn.model_selection import cross_validate # Validación cruzada
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.model_selection import GridSearchCV



# Para preprocesamiento
from sklearn.preprocessing import StandardScaler,MinMaxScaler  # La libreria para reescalar datos
import matplotlib.pyplot as plt


In [77]:
# 1. Cargar datos
datos=pd.read_csv("heart_attack.csv")

## Sobre el conjunto de datos

Age : Edad del paciente. (continua)

Sex : Sexo del paciente (0= male; 1 =female)   (categórica)

exang: angina inducida por el ejercicio (1 = yes; 0 = no)  (categórica)

caa: número de vasos principales del corazón (0-4) (Es ordinal, se manejo como continua) 

cp :Tipo de dolor en el pecho    (categórica)

Value 1: typical angina

Value 2: atypical angina

Value 3: non-anginal pain

Value 4: asymptomatic

trtbps : presión arterial en reposo (in mm Hg) (continua)  

chol : cholestoral in mg/dl fetched via BMI sensor  (continua)

fbs : (glucemia en ayunas > 120 mg/dl) (1 = true; 0 = false)  (categórica)

restecg : resultados electrocardiograma en reposo (categórica)

Value 0: normal

Value 1: having ST-T wave abnormality (T wave inversions and/or ST 
elevation or depression of > 0.05 mV)

Value 2: showing probable or definite left ventricular hypertrophy by 
Estes' criteria

thalach :frecuencia cardíaca máxima alcanzada  (continua)

target : 0= less chance of heart attack 1= more chance of heart attack  (categórica)

In [78]:
datos.head()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,caa,output
0,63,1,3,145,233,1,0,150,0,0,1
1,37,1,2,130,250,0,1,187,0,0,1
2,41,0,1,130,204,0,0,172,0,0,1
3,56,1,1,120,236,0,1,178,0,0,1
4,57,0,0,120,354,0,1,163,1,0,1


In [80]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer

colum_transf=make_column_transformer((OneHotEncoder(),["sex","cp","fbs", "restecg",'exng']),
                                     (StandardScaler(),["age",'trtbps','chol', 'thalachh','caa']))


In [79]:
# target
y=datos['output']

del datos["output"]

#Atributos
X=datos


In [81]:
X.head()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,caa
0,63,1,3,145,233,1,0,150,0,0
1,37,1,2,130,250,0,1,187,0,0
2,41,0,1,130,204,0,0,172,0,0
3,56,1,1,120,236,0,1,178,0,0
4,57,0,0,120,354,0,1,163,1,0


## Aquí va la calibración del hiperparámetro.

In [84]:
# La métrica depende del conjunto de entrenamiento y prueba, 
# hagamos una validación cruzada para que los resultados no dependan de los conjuntos de 
# entrenamiento.

#Definimos los posibles valores del hiperparámetro C. 
espacio_param={"svc__C":[0.01,0.1,0.5,1,1.5,2,5,10,15,20,100]}

# Defino el modelo
pipe=make_pipeline(colum_transf, SVC(kernel="linear"))

# Definir los modelos candidatos
modelo_canditados=GridSearchCV(pipe,param_grid=espacio_param,scoring="accuracy",cv=10,n_jobs=-1)
# Entrenamos los modelos candidatos
modelo_canditados.fit(X,y)

print("El mejor valor del hiperparámetro",modelo_canditados.best_params_)
print("La exactitud promedio es: {:.3f}".format(modelo_canditados.best_score_))

El mejor valor del hiperparámetro {'svc__C': 0.01}
La exactitud promedio es: 0.795


In [ ]:
# Mejoramos la exactitud del modelo... 

In [85]:
modelo_final=SVC(kernel="linear", C=0.01)
# Este modelo lo vamos entrenar empleando TODOS los datos, OJO: Escalar los datos y aplicar 
# OneHotEncoder 
X_std= colum_transf.fit_transform(X)  #>-Todos los datos reescalados.

# Con TODOS los datos reescalados, entrenamos el modelo final
modelo_final.fit(X_std, y)

SVC(C=0.01, kernel='linear')

In [86]:
# Hacer predicciones.
datos_nuevos=pd.DataFrame({"age":[55,60],"sex":[0,1],"cp":[1,2],"trtbps":[140,150],
                           "chol":[220,180],"fbs":[1,1],'restecg':[1,2],
                           "thalachh":[110,118],'exng':[1,0],"caa":[3,4]})
datos_nuevos


,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,caa
0,55,0,1,140,220,1,1,110,1,3
1,60,1,2,150,180,1,2,118,0,4


In [87]:
# Primero estandarizamos
datos_nuevos_std=colum_transf.transform(datos_nuevos)
datos_nuevos_std

array([[ 1.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  1.        ,  0.        ,  1.        ,
         0.        ,  0.        ,  1.        ,  0.06988599,  0.47839125,
        -0.50756498, -1.73377741,  2.22410436],
       [ 0.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ,  1.        ,  0.        ,  0.62133012,  1.04952029,
        -1.28058427, -1.38393337,  3.20361543]])

In [88]:
modelo_final.predict(datos_nuevos_std)

array([0, 0], dtype=int64)

In [75]:
# Estos resultados tienen una exactitud del 0.795

In [ ]:
# el Kernel=?

## Aplicar el truco del Kernel

In [90]:
# La métrica depende del conjunto de entrenamiento y prueba, 
# hagamos una validación cruzada para que los resultados no dependan de los conjuntos de 
# entrenamiento.

#Definimos los posibles valores del hiperparámetro C. 
espacio_param={"svc__C":[0.01,0.1,0.5,1,1.5,2,5,10,15,20,100],
              "svc__gamma":[0.001,0.01,0.5,1,2]}

# Defino el modelo
pipe=make_pipeline(colum_transf, SVC(kernel="rbf"))

# Definir los modelos candidatos
modelo_canditados=GridSearchCV(pipe,param_grid=espacio_param,scoring="accuracy",cv=10,n_jobs=-1)
# Entrenamos los modelos candidatos
modelo_canditados.fit(X,y)

print("El mejor valor del hiperparámetro",modelo_canditados.best_params_)
print("La exactitud promedio es: {:.3f}".format(modelo_canditados.best_score_))

El mejor valor del hiperparámetro {'svc__C': 10, 'svc__gamma': 0.001}
La exactitud promedio es: 0.812


In [ ]:
modelo_final=SVC(kernel="rbf", C=10, gamma=0.001)

In [ ]:
# Bagging ?